## Graph drawing with Bokeh

---

### Get prepared

#### Installations

`py -m pip install bokeh`  
`py -m pip install jupyter_bokeh`  
`py -m pip install selenium`  
geckodriver.exe, add to Path

#### Imports

In [2]:
import math
import networkx as nx
from bokeh.plotting import figure, show
from bokeh.models import GraphRenderer, Ellipse, StaticLayoutProvider
from bokeh.palettes import Spectral8
from bokeh.palettes import Category20_20
from bokeh.plotting import figure, from_networkx, show
from bokeh.models import (BoxSelectTool, HoverTool, MultiLine,
                          NodesAndLinkedEdges, EdgesAndLinkedNodes, Plot, NodesAndAdjacentNodes, Range1d, Scatter, TapTool)
from bokeh.palettes import Spectral4
from bokeh.io import output_notebook, output_file, export_svg
import json

---

### Demo graphs

In [ ]:
# list the nodes and initialize a plot
N = 8
node_indices = list(range(N))

# drawing area
plot = figure(title="Graph layout demonstration", x_range=(-1.1,1.1),
              y_range=(-1.1,1.1), tools="", toolbar_location=None)

graph = GraphRenderer()

# replace the node glyph with an ellipse
# set its height, width, and fill_color
graph.node_renderer.glyph = Ellipse(height=0.1, width=0.2,
                                    fill_color="fill_color")

# assign a palette to ``fill_color`` and add it to the data source
graph.node_renderer.data_source.data = dict(
    index=node_indices,
    fill_color=Spectral8)

# add the rest of the assigned values to the data source
graph.edge_renderer.data_source.data = dict(
    start=[0]*N,
    end=node_indices)

Bokeh comes with a built-in LayoutProvider model that includes a dictionary of (x,y) coordinates for nodes. This lets you arrange plot elements in Cartesian space.

The following codes snippet uses this provider model to produce a plot based on the setup above.

In [ ]:
# generate ellipses based on the node_indices list
circ = [i*2*math.pi/8 for i in node_indices]

# create lists of x- and y-coordinates
x = [math.cos(i) for i in circ]
y = [math.sin(i) for i in circ]

# convert the x and y lists into a dictionary of 2D-coordinates
# and assign each entry to a node on the node_indices list
graph_layout = dict(zip(node_indices, zip(x, y)))

# use the provider model to supply coourdinates to the graph
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)

# render the graph
plot.renderers.append(graph)

# display the plot
show(plot)

By default, the StaticLayoutProvider model draws straight-line paths between the supplied node positions. To set explicit edge paths, supply lists of paths to the bokeh.models.sources.ColumnDataSource data source of the edge_renderer. The StaticLayoutProvider model looks for these paths in the "xs" and "ys" columns of the data source. The paths should be in the same order as the "start" and "end" points. Be extra careful when setting explicit paths because there is no validation to check if they match with node positions.

The following extends the example above and draws quadratic bezier curves between the nodes:

In [ ]:
N = 8
node_indices = list(range(N))

plot = figure(title="Graph Layout Demonstration 2", x_range=(-1.1,1.1), y_range=(-1.1,1.1),
              tools="", toolbar_location=None)

graph = GraphRenderer()

graph.node_renderer.data_source.add(node_indices, 'index')
graph.node_renderer.data_source.add(Spectral8, 'color')
graph.node_renderer.glyph = Ellipse(height=0.1, width=0.2, fill_color="color")

graph.edge_renderer.data_source.data = dict(
    start=[0]*N,
    end=node_indices)

# create a static layout
circ = [i*2*math.pi/8 for i in node_indices]
x = [math.cos(i) for i in circ]
y = [math.sin(i) for i in circ]
graph_layout = dict(zip(node_indices, zip(x, y)))
graph.layout_provider = StaticLayoutProvider(graph_layout=graph_layout)

# draw quadratic bezier paths
def bezier(start, end, control, steps):
    return [(1-s)**2*start + 2*(1-s)*s*control + s**2*end for s in steps]

xs, ys = [], []
sx, sy = graph_layout[0]
steps = [i/100. for i in range(100)]
for node_index in node_indices:
    ex, ey = graph_layout[node_index]
    xs.append(bezier(sx, ex, 0, steps))
    ys.append(bezier(sy, ey, 0, steps))
graph.edge_renderer.data_source.data['xs'] = xs
graph.edge_renderer.data_source.data['ys'] = ys

plot.renderers.append(graph)

show(plot)

---

### NetworkX integration

Bokeh integrates the NetworkX package so you can quickly plot network graphs. The bokeh.plotting.from_networkx convenience method accepts a networkx.Graph object and a NetworkX layout method and returns a configured instance of the GraphRenderer model.


#### Zachary’s karate club graph

Here is how the networkx.spring_layout method lays out the “Zachary’s karate club graph” data set built into NetworkX:

In [ ]:
G = nx.desargues_graph() # always 20 nodes

p = figure(x_range=(-2, 2), y_range=(-2, 2),
           x_axis_location=None, y_axis_location=None,
           tools="hover", tooltips="index: @index")
p.grid.grid_line_color = None

graph = from_networkx(G, nx.spring_layout, scale=1.8, center=(0,0))
p.renderers.append(graph)

# Add some new columns to the node renderer data source
graph.node_renderer.data_source.data['index'] = list(range(len(G)))
graph.node_renderer.data_source.data['colors'] = Category20_20

graph.node_renderer.glyph.update(size=20, fill_color="colors")

show(p)

#### Interaction policies

You can configure the selection or inspection behavior of graphs by setting the selection_policy and inspection_policy attributes of the GraphRenderer. These policy attributes accept a special GraphHitTestPolicy model instance.

For example, setting selection_policy to NodesAndLinkedEdges() lets you select a node and all associated edges. Similarly, setting inspection_policy to EdgesAndLinkedNodes() lets you inspect the "start" and "end" nodes of an edge by hovering over it with the HoverTool. NodesAndAdjacentNodes() lets you inspect a node and all other nodes connected to it by a graph edge.

You can customize the selection_glyph, nonselection_glyph, and/or hover_glyph attributes of the edge and node sub-renderers to add dynamic visual elements to your graph interactions.

Nodes and linked edges

In [ ]:
G = nx.karate_club_graph()

plot = Plot(width=800, height=800, x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
plot.title.text = "Show nodes and linked edges on hover"

plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

graph_renderer = from_networkx(G, nx.circular_layout, scale=1, center=(0, 0))

scatter_glyph = Scatter(size=15, fill_color=Spectral4[0])
graph_renderer.node_renderer.glyph = scatter_glyph
graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[2])
graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[1])

ml_glyph = MultiLine(line_color="#CCCCCC", line_alpha=0.8, line_width=1)
graph_renderer.edge_renderer.glyph = ml_glyph
graph_renderer.edge_renderer.selection_glyph = ml_glyph.clone(line_color=Spectral4[2], line_alpha=1)
graph_renderer.edge_renderer.hover_glyph = ml_glyph.clone(line_color=Spectral4[3], line_width=2)

graph_renderer.selection_policy = NodesAndLinkedEdges()
graph_renderer.inspection_policy = NodesAndLinkedEdges()

plot.renderers.append(graph_renderer)

show(plot)

Edges and linked nodes

In [ ]:
G = nx.karate_club_graph()

plot = Plot(width=1200, height=1200, x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
plot.title.text = "Show edges and linked nodes on hover"

plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

graph_renderer = from_networkx(G, nx.circular_layout, scale=1, center=(0, 0))

scatter_glyph = Scatter(size=15, fill_color=Spectral4[0])
graph_renderer.node_renderer.glyph = scatter_glyph
graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[2])
graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[1])

ml_glyph = MultiLine(line_color="#CCCCCC", line_alpha=0.8, line_width=1)
graph_renderer.edge_renderer.glyph = ml_glyph
graph_renderer.edge_renderer.selection_glyph = ml_glyph.clone(line_color=Spectral4[2], line_alpha=1)
graph_renderer.edge_renderer.hover_glyph = ml_glyph.clone(line_color=Spectral4[3], line_width=1)

graph_renderer.selection_policy = EdgesAndLinkedNodes()
graph_renderer.inspection_policy = EdgesAndLinkedNodes()

plot.renderers.append(graph_renderer)

show(plot)

Nodes and adjacent nodes

In [ ]:
G = nx.karate_club_graph()

plot = Plot(width=1200, height=1200, x_range=Range1d(-1.1, 1.1), y_range=Range1d(-1.1, 1.1))
plot.title.text = "Show nodes and adjacent nodes on hover"

plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

graph_renderer = from_networkx(G, nx.circular_layout, scale=1, center=(0, 0))

scatter_glyph = Scatter(size=15, fill_color=Spectral4[0])
graph_renderer.node_renderer.glyph = scatter_glyph
graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[2])
graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[1])

ml_glyph = MultiLine(line_color="#CCCCCC", line_alpha=0.8, line_width=1)
graph_renderer.edge_renderer.glyph = ml_glyph
graph_renderer.edge_renderer.selection_glyph = ml_glyph.clone(line_color=Spectral4[2], line_alpha=1)
graph_renderer.edge_renderer.hover_glyph = ml_glyph.clone(line_color=Spectral4[2], line_width=1)

graph_renderer.selection_policy = NodesAndAdjacentNodes()
graph_renderer.inspection_policy = NodesAndAdjacentNodes()

plot.renderers.append(graph_renderer)

show(plot)

#### Node and edge attributes

The `from_networkx` method converts node and edge attributes of the NetworkX package for use with `node_renderer` and `edge_renderer` of the GraphRenderer model.

For example, “Zachary’s karate club graph” data set has a node attribute named “club”. You can hover this information with node attributes converted with the `from_networkx` method. You can also use node and edge attributes for color information.

Here is an example of a graph that hovers node attributes and changes colors with edge attributes:

In [ ]:
G = nx.karate_club_graph()

SAME_CLUB_COLOR, DIFFERENT_CLUB_COLOR = "darkgrey", "red"

edge_attrs = {}
for start_node, end_node, _ in G.edges(data=True):
    edge_color = SAME_CLUB_COLOR if G.nodes[start_node]["club"] == G.nodes[end_node]["club"] else DIFFERENT_CLUB_COLOR
    edge_attrs[(start_node, end_node)] = edge_color

nx.set_edge_attributes(G, edge_attrs, "edge_color")

plot = figure(width=1200, height=1200, x_range=(-1.2, 1.2), y_range=(-1.2, 1.2),
              x_axis_location=None, y_axis_location=None, toolbar_location=None,
              title="Graph Interaction Demo", background_fill_color="white",
              tooltips="index: @index, club: @club")
plot.grid.grid_line_color = None

graph_renderer = from_networkx(G, nx.spring_layout, scale=1, center=(0, 0))
graph_renderer.node_renderer.glyph = Scatter(size=15, fill_color="lightblue")
graph_renderer.edge_renderer.glyph = MultiLine(line_color="edge_color",
                                               line_alpha=1, line_width=2)
plot.renderers.append(graph_renderer)

show(plot)

---

### Own experiments

Set node color attribute 

In [ ]:
G = nx.karate_club_graph()

# set edge color based on whether the nodes at either end of the edge belong to the same club or not
SAME_CLUB_COLOR, DIFFERENT_CLUB_COLOR = "darkgrey", "red"

node_attrs = {}
for node, _ in G.nodes(data=True):
    node_color = SAME_CLUB_COLOR if G.degree[node] <= 4 else DIFFERENT_CLUB_COLOR
    node_attrs[node] = node_color
nx.set_node_attributes(G, node_attrs, "node_color")
#print(G.nodes(data=True))

# setup drawing area
plot = figure(width=1200, height=1200, x_range=(-1.2, 1.2), y_range=(-1.2, 1.2),
              x_axis_location=None, y_axis_location=None, toolbar_location=None,
              title="Node color attribute", tooltips="index: @index, club: @club")
plot.grid.grid_line_color = None

#plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

# get node placement from NetworkX spring layout
graph_renderer = from_networkx(G, nx.spring_layout, scale=1, center=(0, 0))

#set node renderer
graph_renderer.node_renderer.glyph = Scatter(size=15, fill_color="node_color")
#graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[3])
#graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[3])

#set edge renderer
graph_renderer.edge_renderer.glyph = MultiLine(line_color="lightblue", line_alpha=1, line_width=2)

#graph_renderer.selection_policy = NodesAndAdjacentNodes()
#graph_renderer.inspection_policy = NodesAndAdjacentNodes()

# render the graph
plot.renderers.append(graph_renderer)
#output_notebook()

show(plot)
plot.output_backend = "svg"
export_svg(plot, filename="graphs/zachary_karate-club.svg")

Combine with hover and select

In [ ]:
G = nx.karate_club_graph()

# set edge color based on whether the nodes at either end of the edge belong to the same club or not
SAME_CLUB_COLOR, DIFFERENT_CLUB_COLOR = "darkgrey", "red"

node_attrs = {}
for node, _ in G.nodes(data=True):
    node_color = SAME_CLUB_COLOR if G.degree[node] <= 4 else DIFFERENT_CLUB_COLOR
    node_attrs[node] = node_color

nx.set_node_attributes(G, node_attrs, "node_color")
#print(G.edges(data=True))

# setup drawing area
plot = figure(width=1200, height=1200, x_range=(-1.2, 1.2), y_range=(-1.2, 1.2),
              x_axis_location=None, y_axis_location=None, toolbar_location=None,
              title="Node color attribute", tooltips="index: @index, club: @club")
plot.grid.grid_line_color = None

plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

# get node placement from NetworkX spring layout
graph_renderer = from_networkx(G, nx.spring_layout, scale=1, center=(0, 0))

#set node renderer
scatter_glyph = Scatter(size=15, fill_color="node_color")
graph_renderer.node_renderer.glyph = scatter_glyph
graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[3])
graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[3])

#set edge renderer
graph_renderer.edge_renderer.glyph = MultiLine(line_color="lightblue", line_alpha=1, line_width=2)

graph_renderer.selection_policy = NodesAndAdjacentNodes()
graph_renderer.inspection_policy = NodesAndAdjacentNodes()

# render the graph
plot.renderers.append(graph_renderer)

show(plot)

Combine with graph export

In [ ]:
G = nx.karate_club_graph()

# set edge color based on whether the nodes at either end of the edge belong to the same club or not
SAME_CLUB_COLOR, DIFFERENT_CLUB_COLOR = "darkgrey", "red"

node_attrs = {}
for node, _ in G.nodes(data=True):
    node_color = SAME_CLUB_COLOR if G.degree[node] <= 4 else DIFFERENT_CLUB_COLOR
    node_attrs[node] = node_color
nx.set_node_attributes(G, node_attrs, "node_color")
#print(G.nodes(data=True))

# setup drawing area
plot = figure(width=1200, height=1200, x_range=(-1.2, 1.2), y_range=(-1.2, 1.2),
              x_axis_location=None, y_axis_location=None, toolbar_location=None,
              title="Node color attribute", tooltips="index: @index, club: @club")
plot.grid.grid_line_color = None

#plot.add_tools(HoverTool(tooltips=None), TapTool(), BoxSelectTool())

# get node placement from NetworkX spring layout
graph_renderer = from_networkx(G, nx.spring_layout, scale=1, center=(0, 0))

#set node renderer
graph_renderer.node_renderer.glyph = Scatter(size=15, fill_color="node_color")
#graph_renderer.node_renderer.selection_glyph = scatter_glyph.clone(fill_color=Spectral4[3])
#graph_renderer.node_renderer.hover_glyph = scatter_glyph.clone(fill_color=Spectral4[3])

#set edge renderer
graph_renderer.edge_renderer.glyph = MultiLine(line_color="lightblue", line_alpha=1, line_width=2)

#graph_renderer.selection_policy = NodesAndAdjacentNodes()
#graph_renderer.inspection_policy = NodesAndAdjacentNodes()

# render the graph
plot.renderers.append(graph_renderer)

show(plot)

---

### Viz for Pres Inaug Addr

#### Read graph in json format

In [29]:
# Reconstruct the graph
with open('graphs/USPresInaugAddr_0.14.json', 'r', encoding='utf-8') as f:
    InaugAddr_json = json.load(f)
G = nx.node_link_graph(InaugAddr_json)
print(G)
#print (G.nodes(data=True))
f=2
p = figure(width=4000, height=4000,x_range=(-f,f), y_range=(-f,f),
           x_axis_location=None, y_axis_location=None,
           tools="hover", tooltips="index: @index")
p.grid.grid_line_color = None

graph = from_networkx(G, nx.spring_layout, scale=5, center=(0,0))
p.renderers.append(graph)

# Add some new columns to the node renderer data source
#graph.node_renderer.data_source.data['index'] = list(range(len(G)))
graph.node_renderer.glyph = Scatter(size=10, fill_color="lightblue")

show(p)

Graph with 153 nodes and 604 edges
